In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Image, display

import models
from datasets import hidden_states
from models import *

In [9]:
def ground_truth(x):
    term1 = -2.0 * x - 1.0
    term2 =  0.5 * x
    term3 =  3.0 * x - 4.0
    return torch.max(torch.max(term1, term2), term3)

x_train = torch.linspace(-3, 3, 200).unsqueeze(1)
y_train = ground_truth(x_train)

model = models.MLP()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)  # Decay LR every 100 epochs

# ---------------------------------------------------------
# 2. Training Loop: Capture frames uniformly
# ---------------------------------------------------------
epochs = 500
capture_interval = 5  # Capture state every 5 epochs
captured_states = []
captured_epochs = []
loss_history = []

for epoch in range(epochs):
    optimizer.zero_grad()

    # capture hidden states
    hidden = model(x_train)
    captured_states.append(hidden.detach().cpu().numpy())

    out = model.output_linear(hidden) # [B, S, 1]
    if not model.classification:
        out = out.squeeze(-1) # [B, S] or [B]

    loss = criterion(out, y_train)
    loss.backward()
    optimizer.step()


    loss_history.append(loss.item())
    captured_states.append(out.detach().numpy())
    captured_epochs.append(epoch)


KeyboardInterrupt



In [ ]:
# ---------------------------------------------------------
# 3. Plot loss history
# ---------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(loss_history, linewidth=2)
plt.title('Training Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
plt.close()

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(8, 5))

# Plot the static target function
ax.plot(x_train.numpy(), y_train.numpy(), 'k--', linewidth=3,
        label='Target Function')

# Initialize the dynamic line for the network's prediction
line, = ax.plot([], [], 'b-', linewidth=2, label='Network Output')
epoch_text = ax.text(0.05, 0.9, '', transform=ax.transAxes, fontsize=12)

ax.set_xlim(-3, 3)
ax.set_ylim(-10, 10) # Adjust based on your target function's range
ax.set_title('Ground Truth vs Model Prediction')
ax.set_xlabel('x')
ax.set_ylabel('F(x)')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='lower right')

# Initialization function for FuncAnimation
def init():
    line.set_data([], [])
    epoch_text.set_text('')
    return line, epoch_text

# Update function called for every frame
def update(frame_idx):
    # Update the network's line
    line.set_data(x_train.numpy(), captured_states[frame_idx])
    # Update the epoch counter text
    epoch_text.set_text(f'Epoch: {captured_epochs[frame_idx]}')
    return line, epoch_text

# Generate the animation
anim = FuncAnimation(
    fig,
    update,
    frames=len(captured_states),
    init_func=init,
    blit=True,
    interval=50 # ms between frames
)

# ---------------------------------------------------------
# 4. Save as GIF and Display in Notebook
# ---------------------------------------------------------
gif_path = 'conv_animation.gif'
# Requires the 'pillow' library (usually pre-installed in Jupyter environments)
anim.save(gif_path, writer='pillow', fps=15)

plt.close(fig)

# Render the saved GIF inside the notebook
display(Image(filename=gif_path))